# Phase 3: Alpha Research & Statistical Learning  
## Notebook 03_05 — Vector Autoregression and Granger Causality

### Research question

Can lagged cross-asset information improve forecasts, and how can Vector Autoregression and Granger causality tests be used without confusing predictive causality with true economic causality?

This notebook follows naturally from:

```text
03_01_garch_volatility_modeling
03_02_har_realized_volatility_forecasting
03_03_markov_regime_switching
03_04_gmm_hmm_tick_regime_detection
```

The previous notebooks modelled volatility and hidden regimes. This notebook studies **lead-lag relationships** across time series.

It covers:

1. Vector Autoregression, VAR($p$);
2. stationarity and return transformations;
3. lagged design matrices;
4. OLS estimation from scratch;
5. lag-order selection using AIC/BIC;
6. Granger causality tests with restricted/unrestricted regressions;
7. F-tests for lag exclusion;
8. synthetic cross-asset data with known lead-lag structure;
9. train/test forecasting;
10. VAR versus univariate autoregressive baselines;
11. impulse response functions;
12. forecast error variance decomposition;
13. rolling Granger stability diagnostics;
14. limitations and research-frontier extensions.

The key idea:

> A variable Granger-causes another variable if its past values improve forecasts after controlling for the target's own past. This is predictive causality, not proof of structural economic causality.

## 1. VAR model

For a vector of stationary time series:

$$
y_t =
\begin{bmatrix}
y_{1,t}\\
y_{2,t}\\
\vdots\\
y_{K,t}
\end{bmatrix}
$$

a VAR($p$) model is:

$$
\begin{aligned}
y_t &= c \\
&\quad + A_1y_{t-1} \\
&\quad + A_2y_{t-2} \\
&\quad + \cdots \\
&\quad + A_py_{t-p} \\
&\quad + \varepsilon_t
\end{aligned}
$$

where:

- $c$ is a vector of intercepts;
- $A_i$ are coefficient matrices;
- $\varepsilon_t$ is a vector of innovations.

Each variable is regressed on lags of itself and all other variables.

For finance, VAR is useful for:

- lead-lag diagnostics;
- macro/market interaction studies;
- futures vs spot relationships;
- cross-asset prediction;
- risk spillover analysis;
- impulse response analysis.

## 2. Granger causality

A variable $x_t$ Granger-causes $y_t$ if lags of $x$ improve forecasts of $y$ after controlling for lags of $y$.

Restricted model:

$$
\begin{aligned}
y_t &= \alpha \\
&\quad + \sum_{i=1}^{p}\phi_i y_{t-i} \\
&\quad + u_t
\end{aligned}
$$

Unrestricted model:

$$
\begin{aligned}
y_t &= \alpha \\
&\quad + \sum_{i=1}^{p}\phi_i y_{t-i} \\
&\quad + \sum_{i=1}^{p}\beta_i x_{t-i} \\
&\quad + e_t
\end{aligned}
$$

The null hypothesis is:

$$
H_0: \beta_1=\beta_2=\cdots=\beta_p=0
$$

If we reject $H_0$, then $x$ has incremental predictive content for $y$.

Important warning:

> Granger causality is about forecasting. It does not prove true economic causality, mechanism, or arbitrage.

## 3. Stationarity warning

VAR models are usually applied to stationary series.

Prices are often non-stationary.

Returns are usually more suitable:

$$
r_t = \log P_t - \log P_{t-1}
$$

This notebook works with returns and stationary synthetic processes.

If using real data:

1. check for unit roots;
2. use log returns or differences;
3. consider cointegration and VECM if prices share long-run relationships.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy import stats
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

SCIPY_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class VARConfig:
    n_obs: int = 2_500
    train_fraction: float = 0.70
    max_lag: int = 8
    selected_lag_default: int = 3
    rolling_window: int = 750
    rolling_step: int = 50
    seed: int = 42


config = VARConfig()
config

## 5. Synthetic cross-asset return data

We simulate a small cross-asset system with known lead-lag relationships.

Variables:

| Variable | Interpretation |
|---|---|
| `futures_ret` | liquid futures return |
| `spot_ret` | slower spot/cash proxy |
| `fx_ret` | currency move |
| `commodity_ret` | commodity-sensitive asset |
| `risk_ret` | broad risk factor |

Built-in predictive structure:

1. futures lags predict spot;
2. FX lags predict commodity;
3. broad risk lags affect both futures and commodity;
4. spot has weaker feedback into futures.

This lets us test whether Granger diagnostics recover the planted structure.

In [ ]:
def simulate_cross_asset_var(config: VARConfig) -> tuple[pd.DataFrame, dict]:
    rng = np.random.default_rng(config.seed)
    n = config.n_obs

    cols = ["futures_ret", "spot_ret", "fx_ret", "commodity_ret", "risk_ret"]
    K = len(cols)

    # VAR(2) coefficient matrices.
    A1 = np.array([
        [ 0.08,  0.02,  0.00,  0.02,  0.12],  # futures
        [ 0.18,  0.05,  0.00,  0.01,  0.05],  # spot: futures leads spot
        [ 0.00,  0.00,  0.06,  0.00, -0.05],  # fx
        [ 0.03,  0.00, -0.14,  0.06,  0.10],  # commodity: fx and risk lead
        [ 0.02,  0.00,  0.00,  0.01,  0.10],  # risk
    ])

    A2 = np.array([
        [ 0.02,  0.00,  0.00,  0.00,  0.04],
        [ 0.10,  0.03,  0.00,  0.00,  0.02],
        [ 0.00,  0.00,  0.03,  0.00, -0.02],
        [ 0.01,  0.00, -0.08,  0.03,  0.04],
        [ 0.00,  0.00,  0.00,  0.00,  0.04],
    ])

    intercept = np.array([0.00010, 0.00008, 0.00000, 0.00005, 0.00006])

    # Correlated innovation covariance.
    base_vol = np.array([0.010, 0.008, 0.006, 0.012, 0.009])
    corr = np.array([
        [1.00, 0.45, 0.05, 0.25, 0.45],
        [0.45, 1.00, 0.03, 0.20, 0.35],
        [0.05, 0.03, 1.00, -0.30, -0.15],
        [0.25, 0.20, -0.30, 1.00, 0.40],
        [0.45, 0.35, -0.15, 0.40, 1.00],
    ])

    cov = np.outer(base_vol, base_vol) * corr

    y = np.zeros((n, K))
    shocks = rng.multivariate_normal(np.zeros(K), cov, size=n)

    for t in range(2, n):
        y[t] = intercept + A1 @ y[t - 1] + A2 @ y[t - 2] + shocks[t]

    dates = pd.bdate_range("2015-01-01", periods=n)

    df = pd.DataFrame(y, columns=cols)
    df.insert(0, "timestamp", dates)

    for col in cols:
        df[col.replace("_ret", "_price")] = 100 * np.exp(df[col].cumsum())

    truth = {
        "columns": cols,
        "A1": A1,
        "A2": A2,
        "intercept": intercept,
        "innovation_cov": cov
    }

    return df, truth


data, truth = simulate_cross_asset_var(config)

data.head()

In [ ]:
price_cols = [c for c in data.columns if c.endswith("_price")]
return_cols = truth["columns"]

plt.figure(figsize=(12, 6))
for col in price_cols:
    plt.plot(data["timestamp"], data[col], label=col)
plt.title("Synthetic Cross-Asset Price Series")
plt.xlabel("Date")
plt.ylabel("Synthetic price")
plt.legend()
plt.show()

plt.figure(figsize=(12, 6))
for col in return_cols:
    plt.plot(data["timestamp"], data[col], label=col, alpha=0.8)
plt.title("Synthetic Cross-Asset Returns")
plt.xlabel("Date")
plt.ylabel("Return")
plt.legend()
plt.show()

## 6. Basic diagnostics

We check:

1. mean and volatility;
2. return correlation;
3. autocorrelation.

VAR models are about lagged interactions, not just contemporaneous correlation.

In [ ]:
summary_stats = (
    data[return_cols]
    .agg(["mean", "std", "skew"])
    .T
    .rename(columns={"mean": "daily_mean", "std": "daily_vol", "skew": "skew"})
)

summary_stats["annualised_mean"] = summary_stats["daily_mean"] * 252
summary_stats["annualised_vol"] = summary_stats["daily_vol"] * np.sqrt(252)

summary_stats

In [ ]:
corr_matrix = data[return_cols].corr()

corr_matrix

In [ ]:
plt.figure(figsize=(7, 6))
plt.imshow(corr_matrix.values, vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(return_cols)), return_cols, rotation=45, ha="right")
plt.yticks(range(len(return_cols)), return_cols)
plt.title("Return Correlation Matrix")
plt.tight_layout()
plt.show()

## 7. Lagged design matrix

For VAR($p$), the design matrix contains:

$$
1,\quad y_{t-1},\quad y_{t-2},\dots,y_{t-p}
$$

The target matrix is:

$$
Y_t
$$

We implement this manually.

In [ ]:
def make_var_design(df: pd.DataFrame, cols: list[str], lag: int) -> tuple[np.ndarray, np.ndarray, pd.DataFrame, list[str]]:
    y = df[cols].to_numpy()
    rows = []
    targets = []
    timestamps = []

    feature_names = ["intercept"]

    for l in range(1, lag + 1):
        for col in cols:
            feature_names.append(f"{col}_lag{l}")

    for t in range(lag, len(df)):
        x_t = [1.0]
        for l in range(1, lag + 1):
            x_t.extend(y[t - l].tolist())

        rows.append(x_t)
        targets.append(y[t])
        timestamps.append(df["timestamp"].iloc[t])

    X = np.asarray(rows, dtype=float)
    Y = np.asarray(targets, dtype=float)
    meta = pd.DataFrame({"timestamp": timestamps})

    return X, Y, meta, feature_names


X_demo, Y_demo, meta_demo, feature_names_demo = make_var_design(data, return_cols, lag=2)

X_demo.shape, Y_demo.shape, feature_names_demo[:8]

## 8. OLS VAR estimation

Each equation in a VAR can be estimated by OLS.

For all equations together:

$$
\hat B=(X^\top X)^{-1}X^\top Y
$$

where $B$ contains coefficients for all target variables.

Residual covariance:

$$
\hat\Sigma_\varepsilon = \frac{E^\top E}{T-k}
$$

In [ ]:
def fit_var_ols(df: pd.DataFrame, cols: list[str], lag: int, ridge: float = 1e-10) -> dict:
    X, Y, meta, feature_names = make_var_design(df, cols, lag)

    xtx = X.T @ X
    xty = X.T @ Y

    try:
        beta = np.linalg.solve(xtx, xty)
    except np.linalg.LinAlgError:
        beta = np.linalg.solve(xtx + ridge * np.eye(X.shape[1]), xty)

    fitted = X @ beta
    residuals = Y - fitted

    T_eff, n_features = X.shape
    K = Y.shape[1]

    sigma = residuals.T @ residuals / max(T_eff - n_features, 1)

    return {
        "lag": lag,
        "cols": cols,
        "X": X,
        "Y": Y,
        "meta": meta,
        "feature_names": feature_names,
        "beta": beta,
        "fitted": fitted,
        "residuals": residuals,
        "sigma": sigma,
        "T_eff": T_eff,
        "n_features": n_features,
        "K": K
    }


var2_fit = fit_var_ols(data, return_cols, lag=2)

pd.DataFrame(var2_fit["beta"], index=var2_fit["feature_names"], columns=return_cols).head(12)

## 9. Lag-order selection

Information criteria balance fit and complexity.

For residual covariance matrix $\hat\Sigma$:

$$
AIC(p)=\log|\hat\Sigma_p|+\frac{2m}{T}
$$

$$
BIC(p)=\log|\hat\Sigma_p|+\frac{\log(T)m}{T}
$$

where $m$ is the number of estimated parameters across equations.

Lower is better.

In [ ]:
def lag_order_selection(df: pd.DataFrame, cols: list[str], max_lag: int) -> pd.DataFrame:
    rows = []

    for lag in range(1, max_lag + 1):
        fit = fit_var_ols(df, cols, lag)
        sigma = fit["sigma"]

        sign, logdet = np.linalg.slogdet(sigma)
        if sign <= 0:
            logdet = np.log(np.linalg.det(sigma + 1e-10 * np.eye(len(cols))))

        K = len(cols)
        n_params_per_eq = 1 + K * lag
        total_params = K * n_params_per_eq
        T_eff = fit["T_eff"]

        rows.append({
            "lag": lag,
            "logdet_sigma": float(logdet),
            "aic": float(logdet + 2 * total_params / T_eff),
            "bic": float(logdet + np.log(T_eff) * total_params / T_eff),
            "total_params": total_params,
            "T_eff": T_eff
        })

    return pd.DataFrame(rows)


lag_selection = lag_order_selection(data, return_cols, config.max_lag)

lag_selection

In [ ]:
selected_lag_aic = int(lag_selection.sort_values("aic").iloc[0]["lag"])
selected_lag_bic = int(lag_selection.sort_values("bic").iloc[0]["lag"])

pd.Series({
    "selected_lag_aic": selected_lag_aic,
    "selected_lag_bic": selected_lag_bic,
    "default_selected_lag_used": config.selected_lag_default
})

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(lag_selection["lag"], lag_selection["aic"], marker="o", label="AIC")
plt.plot(lag_selection["lag"], lag_selection["bic"], marker="x", label="BIC")
plt.title("VAR Lag Order Selection")
plt.xlabel("Lag")
plt.ylabel("Information criterion")
plt.legend()
plt.show()

## 10. Train/test split

We estimate models on a training period and evaluate one-step forecasts on a test period.

No shuffling is used.

In [ ]:
split_idx = int(len(data) * config.train_fraction)

train_data = data.iloc[:split_idx].copy()
test_data = data.iloc[split_idx:].copy()

selected_lag = config.selected_lag_default

var_fit = fit_var_ols(train_data, return_cols, selected_lag)

pd.Series({
    "selected_lag": selected_lag,
    "n_train": len(train_data),
    "n_test": len(test_data),
    "train_start": train_data["timestamp"].iloc[0],
    "train_end": train_data["timestamp"].iloc[-1],
    "test_start": test_data["timestamp"].iloc[0],
    "test_end": test_data["timestamp"].iloc[-1]
})

## 11. VAR coefficient inspection

We inspect the fitted coefficient matrices:

$$
A_1,\dots,A_p
$$

The row is the target variable; the column is the predictor variable at a given lag.

In [ ]:
def extract_var_matrices(fit: dict) -> tuple[np.ndarray, np.ndarray]:
    beta = fit["beta"]
    K = fit["K"]
    lag = fit["lag"]

    intercept = beta[0, :]
    matrices = []

    row_start = 1
    for l in range(lag):
        block = beta[row_start: row_start + K, :].T
        matrices.append(block)
        row_start += K

    return intercept, np.array(matrices)


intercept_hat, A_hat = extract_var_matrices(var_fit)

coef_tables = []

for l in range(selected_lag):
    table = pd.DataFrame(A_hat[l], index=return_cols, columns=return_cols)
    table["lag"] = l + 1
    coef_tables.append(table.reset_index().rename(columns={"index": "target"}))

coef_table = pd.concat(coef_tables, ignore_index=True)

pd.DataFrame(A_hat[0], index=return_cols, columns=return_cols)

In [ ]:
pd.DataFrame(A_hat[1], index=return_cols, columns=return_cols)

## 12. One-step VAR forecasting

One-step forecast:

$$
\begin{aligned}
\hat y_{t+1} &= \hat c \\
&\quad + \hat A_1y_t \\
&\quad + \cdots \\
&\quad + \hat A_py_{t-p+1}
\end{aligned}
$$

We build forecasts recursively using observed lagged test data, not predicted future values.

In [ ]:
def one_step_var_forecasts(train_df, test_df, cols, fit):
    lag = fit["lag"]
    beta = fit["beta"]
    feature_names = fit["feature_names"]

    combined = pd.concat([train_df, test_df], ignore_index=True)
    start_idx = len(train_df)

    rows = []

    y_all = combined[cols].to_numpy()

    for idx in range(start_idx, len(combined)):
        if idx < lag:
            continue

        x = [1.0]
        for l in range(1, lag + 1):
            x.extend(y_all[idx - l].tolist())

        x = np.asarray(x)
        pred = x @ beta
        actual = y_all[idx]

        row = {"timestamp": combined["timestamp"].iloc[idx]}
        for j, col in enumerate(cols):
            row[f"actual_{col}"] = actual[j]
            row[f"forecast_var_{col}"] = pred[j]
            row[f"error_var_{col}"] = pred[j] - actual[j]

        rows.append(row)

    return pd.DataFrame(rows)


var_forecasts = one_step_var_forecasts(train_data, test_data, return_cols, var_fit)

var_forecasts.head()

## 13. Univariate AR baseline

A VAR only matters if cross-variable lags improve forecasts.

We compare against separate univariate AR($p$) models:

$$
\begin{aligned}
y_{i,t} &= c_i \\
&\quad + \sum_{\ell=1}^{p}\phi_{i,\ell}y_{i,t-\ell} \\
&\quad + \epsilon_{i,t}
\end{aligned}
$$

In [ ]:
def fit_univariate_ar(train_df, col, lag):
    y = train_df[col].to_numpy()
    X = []
    target = []

    for t in range(lag, len(y)):
        X.append([1.0] + [y[t - l] for l in range(1, lag + 1)])
        target.append(y[t])

    X = np.asarray(X)
    target = np.asarray(target)

    beta = np.linalg.pinv(X.T @ X) @ X.T @ target

    return beta


def one_step_ar_forecasts(train_df, test_df, cols, lag):
    combined = pd.concat([train_df, test_df], ignore_index=True)
    start_idx = len(train_df)

    betas = {col: fit_univariate_ar(train_df, col, lag) for col in cols}
    y_all = {col: combined[col].to_numpy() for col in cols}

    rows = []

    for idx in range(start_idx, len(combined)):
        row = {"timestamp": combined["timestamp"].iloc[idx]}

        for col in cols:
            x = np.array([1.0] + [y_all[col][idx - l] for l in range(1, lag + 1)])
            pred = float(x @ betas[col])
            actual = float(y_all[col][idx])

            row[f"actual_{col}"] = actual
            row[f"forecast_ar_{col}"] = pred
            row[f"error_ar_{col}"] = pred - actual

        rows.append(row)

    return pd.DataFrame(rows)


ar_forecasts = one_step_ar_forecasts(train_data, test_data, return_cols, selected_lag)

forecast_comparison = var_forecasts.merge(ar_forecasts, on=["timestamp"] + [f"actual_{c}" for c in return_cols], how="inner")

forecast_comparison.head()

## 14. Forecast evaluation

We compute for each target:

- MSE;
- MAE;
- directional accuracy;
- correlation between forecast and actual.

For directional accuracy:

$$
\operatorname{sign}(\hat y_t)=\operatorname{sign}(y_t)
$$

In [ ]:
def forecast_metrics_by_target(forecasts, cols):
    rows = []

    for col in cols:
        actual = forecasts[f"actual_{col}"].to_numpy()

        for method in ["var", "ar"]:
            pred = forecasts[f"forecast_{method}_{col}"].to_numpy()
            error = pred - actual

            rows.append({
                "target": col,
                "method": method,
                "mse": float(np.mean(error ** 2)),
                "mae": float(np.mean(np.abs(error))),
                "forecast_actual_corr": float(np.corrcoef(pred, actual)[0, 1]),
                "directional_accuracy": float(np.mean(np.sign(pred) == np.sign(actual)))
            })

    return pd.DataFrame(rows)


forecast_metrics = forecast_metrics_by_target(forecast_comparison, return_cols)

forecast_metrics.sort_values(["target", "mse"])

In [ ]:
plt.figure(figsize=(12, 6))
plot_df = forecast_metrics.copy()
plot_df["label"] = plot_df["target"] + "_" + plot_df["method"]
plt.barh(plot_df["label"], plot_df["mse"])
plt.title("Forecast MSE by Target and Method")
plt.xlabel("MSE")
plt.ylabel("Target/method")
plt.show()

## 15. Granger causality test from restricted and unrestricted regressions

For target $y$, source $x$, and lag $p$:

- restricted model uses lags of all variables except source $x$;
- unrestricted model uses lags of all variables.

The F-statistic is:

$$
F = \frac{(SSR_r-SSR_u)/q} {SSR_u/(T-k_u)}
$$

where:

- $SSR_r$ is restricted sum of squared residuals;
- $SSR_u$ is unrestricted sum of squared residuals;
- $q$ is the number of excluded lag coefficients;
- $k_u$ is the number of unrestricted parameters.

If SciPy is available, we compute p-values from the F distribution.

In [ ]:
def granger_test_pair(df, cols, target, source, lag):
    X_full, Y, meta, feature_names = make_var_design(df, cols, lag)
    target_idx = cols.index(target)

    y = Y[:, target_idx]

    # Unrestricted: all lags of all variables.
    X_u = X_full

    # Restricted: remove lagged source columns.
    source_feature_names = {f"{source}_lag{l}" for l in range(1, lag + 1)}
    keep_idx = [i for i, name in enumerate(feature_names) if name not in source_feature_names]
    X_r = X_full[:, keep_idx]

    beta_u = np.linalg.pinv(X_u.T @ X_u) @ X_u.T @ y
    beta_r = np.linalg.pinv(X_r.T @ X_r) @ X_r.T @ y

    resid_u = y - X_u @ beta_u
    resid_r = y - X_r @ beta_r

    ssr_u = float(resid_u @ resid_u)
    ssr_r = float(resid_r @ resid_r)

    q = X_u.shape[1] - X_r.shape[1]
    df_den = X_u.shape[0] - X_u.shape[1]

    numerator = (ssr_r - ssr_u) / max(q, 1)
    denominator = ssr_u / max(df_den, 1)

    F = numerator / denominator if denominator > 0 else np.nan

    if SCIPY_AVAILABLE and np.isfinite(F):
        p_value = float(1 - stats.f.cdf(F, q, df_den))
    else:
        p_value = np.nan

    return {
        "target": target,
        "source": source,
        "lag": lag,
        "ssr_restricted": ssr_r,
        "ssr_unrestricted": ssr_u,
        "F_stat": float(F),
        "df_num": int(q),
        "df_den": int(df_den),
        "p_value": p_value
    }


def granger_matrix(df, cols, lag):
    rows = []

    for target in cols:
        for source in cols:
            if source == target:
                continue
            rows.append(granger_test_pair(df, cols, target, source, lag))

    return pd.DataFrame(rows)


granger_results = granger_matrix(train_data, return_cols, selected_lag)

granger_results.sort_values("p_value").head(15)

## 16. Granger causality heatmap

Rows are targets.

Columns are sources.

Each cell answers:

> Does the source help forecast the target?

We visualise:

$$
-\log_{10}(p\text{-value})
$$

Larger values indicate stronger evidence against the null of no Granger causality.

In [ ]:
granger_pivot = granger_results.pivot(index="target", columns="source", values="p_value")
neglog_p = -np.log10(granger_pivot.clip(lower=1e-12))

plt.figure(figsize=(8, 6))
plt.imshow(neglog_p.values, aspect="auto")
plt.colorbar(label="-log10(p-value)")
plt.xticks(range(len(neglog_p.columns)), neglog_p.columns, rotation=45, ha="right")
plt.yticks(range(len(neglog_p.index)), neglog_p.index)
plt.title("Granger Causality Heatmap")
plt.tight_layout()
plt.show()

granger_pivot

## 17. Known planted relationships

The synthetic data was designed with these important lead-lag effects:

1. `futures_ret` leads `spot_ret`;
2. `fx_ret` leads `commodity_ret`;
3. `risk_ret` leads `futures_ret`;
4. `risk_ret` leads `commodity_ret`.

We check whether Granger tests detect these relationships.

In [ ]:
planted_pairs = pd.DataFrame([
    {"source": "futures_ret", "target": "spot_ret", "relationship": "futures leads spot"},
    {"source": "fx_ret", "target": "commodity_ret", "relationship": "FX leads commodity"},
    {"source": "risk_ret", "target": "futures_ret", "relationship": "risk leads futures"},
    {"source": "risk_ret", "target": "commodity_ret", "relationship": "risk leads commodity"},
])

planted_check = planted_pairs.merge(
    granger_results,
    on=["source", "target"],
    how="left"
)

planted_check[["relationship", "source", "target", "F_stat", "p_value"]]

## 18. Rolling Granger stability

A relationship that appears once may be unstable.

We test Granger causality over rolling windows.

This helps detect:

- regime dependence;
- structural breaks;
- unstable relationships;
- data-mined false positives.

In [ ]:
def rolling_granger(df, cols, source, target, lag, window, step):
    rows = []

    for start in range(0, len(df) - window + 1, step):
        sub = df.iloc[start:start + window]
        result = granger_test_pair(sub, cols, target=target, source=source, lag=lag)

        rows.append({
            "window_start": sub["timestamp"].iloc[0],
            "window_end": sub["timestamp"].iloc[-1],
            "source": source,
            "target": target,
            "F_stat": result["F_stat"],
            "p_value": result["p_value"]
        })

    return pd.DataFrame(rows)


rolling_pairs = [
    ("futures_ret", "spot_ret"),
    ("fx_ret", "commodity_ret"),
    ("risk_ret", "futures_ret"),
    ("risk_ret", "commodity_ret")
]

rolling_frames = []

for source, target in rolling_pairs:
    rolling_frames.append(
        rolling_granger(
            data,
            return_cols,
            source=source,
            target=target,
            lag=selected_lag,
            window=config.rolling_window,
            step=config.rolling_step
        )
    )

rolling_granger_results = pd.concat(rolling_frames, ignore_index=True)

rolling_granger_results.head()

In [ ]:
plt.figure(figsize=(12, 6))

for (source, target), group in rolling_granger_results.groupby(["source", "target"]):
    label = f"{source} -> {target}"
    plt.plot(group["window_end"], -np.log10(group["p_value"].clip(lower=1e-12)), marker="o", label=label)

plt.axhline(-np.log10(0.05), linestyle="--", label="5% threshold")
plt.title("Rolling Granger Significance")
plt.xlabel("Window end")
plt.ylabel("-log10(p-value)")
plt.legend()
plt.show()

## 19. Impulse response functions

A VAR can trace how a shock propagates through the system.

Ignoring orthogonalisation for simplicity, the impulse response at horizon $h$ answers:

> If one variable receives a one-unit shock today, how do future variables respond under the fitted VAR dynamics?

For a VAR($p$), we simulate recursively from an initial shock vector.

In [ ]:
def var_impulse_response(A_matrices, shock_index, horizon, shock_size=1.0):
    """
    Non-orthogonal impulse response for VAR(p).
    """
    p, K, _ = A_matrices.shape

    history = [np.zeros(K) for _ in range(p)]
    history[-1][shock_index] = shock_size

    responses = []
    responses.append(history[-1].copy())

    for h in range(1, horizon + 1):
        y_next = np.zeros(K)

        for lag_idx in range(p):
            y_next += A_matrices[lag_idx] @ history[-1 - lag_idx]

        responses.append(y_next.copy())
        history.append(y_next)
        history = history[-p:]

    return np.array(responses)


horizon = 20
shock_source = "futures_ret"
shock_idx = return_cols.index(shock_source)

irf = var_impulse_response(A_hat, shock_idx, horizon=horizon, shock_size=0.01)

irf_df = pd.DataFrame(irf, columns=return_cols)
irf_df.insert(0, "horizon", range(horizon + 1))

irf_df.head()

In [ ]:
plt.figure(figsize=(12, 6))
for col in return_cols:
    plt.plot(irf_df["horizon"], irf_df[col], marker="o", label=col)
plt.axhline(0, linestyle="--")
plt.title(f"Impulse Response to Shock in {shock_source}")
plt.xlabel("Horizon")
plt.ylabel("Response")
plt.legend()
plt.show()

## 20. Forecast error variance decomposition, simplified

A simplified FEVD asks:

> Over a forecast horizon, what fraction of response energy comes from each shock?

This implementation uses non-orthogonal unit shocks and is illustrative only.

Production FEVD usually uses orthogonalised shocks based on the innovation covariance matrix.

In [ ]:
def simplified_fevd(A_matrices, horizon):
    p, K, _ = A_matrices.shape
    contributions = np.zeros((K, K))

    for shock_idx in range(K):
        irf = var_impulse_response(A_matrices, shock_idx, horizon=horizon, shock_size=1.0)
        contributions[:, shock_idx] = np.sum(irf ** 2, axis=0)

    row_sums = contributions.sum(axis=1, keepdims=True)
    shares = contributions / np.maximum(row_sums, 1e-12)

    return pd.DataFrame(shares, index=return_cols, columns=[f"shock_{c}" for c in return_cols])


fevd_simple = simplified_fevd(A_hat, horizon=20)

fevd_simple

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(fevd_simple.values, aspect="auto")
plt.colorbar(label="Share")
plt.xticks(range(len(fevd_simple.columns)), fevd_simple.columns, rotation=45, ha="right")
plt.yticks(range(len(fevd_simple.index)), fevd_simple.index)
plt.title("Simplified Forecast Error Variance Decomposition")
plt.tight_layout()
plt.show()

## 21. Practical checklist for VAR and Granger research

Before trusting VAR/Granger results, check:

1. **Stationarity**  
   Are variables stationary? Use returns/differences if needed.

2. **Lag selection**  
   Does the result depend heavily on lag order?

3. **Train/test evaluation**  
   Does VAR improve out-of-sample forecasts?

4. **Multiple testing**  
   Many pairwise tests produce false positives.

5. **Contemporaneous correlation**  
   Granger tests are about lagged prediction, not same-time correlation.

6. **Economic mechanism**  
   Is there a plausible reason for the lead-lag effect?

7. **Regime stability**  
   Is the relationship stable through time?

8. **Data alignment**  
   Are timestamps, holidays, sessions, and close times aligned?

9. **Execution feasibility**  
   Can the signal be acted on after costs and latency?

10. **Causality language**  
   Say "predictive content" unless you have structural identification.

## 22. Saving outputs

The notebook saves:

1. synthetic cross-asset data;
2. summary statistics;
3. lag-order selection;
4. fitted VAR coefficients;
5. forecast comparison table;
6. Granger causality results;
7. planted relationship checks;
8. rolling Granger diagnostics;
9. impulse response table;
10. simplified FEVD table;
11. manifest.

In [ ]:
output_dir = Path("data/processed/vector_autoregression_granger_v1")
output_dir.mkdir(parents=True, exist_ok=True)

data_path = output_dir / "synthetic_cross_asset_returns.csv"
summary_path = output_dir / "summary_statistics.csv"
corr_path = output_dir / "return_correlation_matrix.csv"
lag_selection_path = output_dir / "lag_order_selection.csv"
coef_path = output_dir / "var_coefficients.csv"
forecast_path = output_dir / "forecast_comparison.csv"
metrics_path = output_dir / "forecast_metrics.csv"
granger_path = output_dir / "granger_causality_results.csv"
planted_path = output_dir / "planted_relationship_check.csv"
rolling_path = output_dir / "rolling_granger_results.csv"
irf_path = output_dir / "impulse_response.csv"
fevd_path = output_dir / "simplified_fevd.csv"
manifest_path = output_dir / "manifest.json"

data.to_csv(data_path, index=False)
summary_stats.to_csv(summary_path)
corr_matrix.to_csv(corr_path)
lag_selection.to_csv(lag_selection_path, index=False)
coef_table.to_csv(coef_path, index=False)
forecast_comparison.to_csv(forecast_path, index=False)
forecast_metrics.to_csv(metrics_path, index=False)
granger_results.to_csv(granger_path, index=False)
planted_check.to_csv(planted_path, index=False)
rolling_granger_results.to_csv(rolling_path, index=False)
irf_df.to_csv(irf_path, index=False)
fevd_simple.to_csv(fevd_path)

manifest = {
    "dataset_name": "vector_autoregression_granger_outputs",
    "schema_version": "vector_autoregression_granger_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "return_cols": return_cols,
    "selected_lag": selected_lag,
    "selected_lag_aic": selected_lag_aic,
    "selected_lag_bic": selected_lag_bic,
    "scipy_available": SCIPY_AVAILABLE,
    "best_forecasts_by_mse": forecast_metrics.sort_values("mse").groupby("target").head(1).to_dict(orient="records"),
    "planted_relationship_check": planted_check[["relationship", "source", "target", "F_stat", "p_value"]].to_dict(orient="records"),
    "notes": [
        "Synthetic data is generated from a stationary VAR(2) with known planted lead-lag structure.",
        "VAR is estimated by OLS from scratch.",
        "Granger causality tests compare restricted and unrestricted regressions.",
        "P-values require SciPy; F-statistics are still computed without SciPy.",
        "Impulse responses are non-orthogonal and illustrative.",
        "Granger causality means predictive content, not structural economic causality."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

data_path, lag_selection_path, granger_path, forecast_path, manifest_path

## 23. Limitations

### 23.1 Synthetic data

The notebook uses synthetic data with known lead-lag relationships.

Real data has changing relationships, outliers, holidays, missing observations, session boundaries, and contract rolls.

### 23.2 Linear model

VAR is linear.

Cross-asset relationships may be nonlinear or regime-dependent.

### 23.3 Stationarity assumption

VAR should be applied to stationary series.

If price levels are cointegrated, a Vector Error Correction Model may be more appropriate.

### 23.4 Granger causality is not structural causality

A Granger relationship may arise from:

- common omitted variables;
- asynchronous timestamps;
- stale prices;
- liquidity differences;
- market microstructure delays;
- data-mining.

### 23.5 Multiple testing

Testing every pair creates many p-values.

False discovery control may be needed.

### 23.6 Lag-order sensitivity

Results can change with lag length.

Lag selection should be robust.

### 23.7 Simplified impulse response

The impulse response and FEVD here are non-orthogonal.

Production macro/VAR analysis often uses Cholesky or structural identification.

## 24. Research frontier and extensions

Important extensions include:

1. **VECM**  
   For cointegrated price levels.

2. **Structural VAR**  
   Use identification assumptions for economic shocks.

3. **Time-varying parameter VAR**  
   Allow relationships to change over time.

4. **Bayesian VAR**  
   Stabilise high-dimensional VARs with priors.

5. **Sparse VAR / LASSO VAR**  
   Handle many assets without overfitting.

6. **Regime-switching VAR**  
   Combine HMM regimes with VAR dynamics.

7. **Nonlinear Granger causality**  
   Use tree models, kernels, or neural networks.

8. **High-frequency lead-lag VAR**  
   Align tick/bar data carefully and account for asynchronous trading.

9. **Transfer entropy**  
   Nonlinear directional dependence measure.

10. **Chinese futures cross-market lead-lag**  
   Study night sessions, overseas leads, sector chains, and contract-roll effects.

## 25. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_06_factor_momentum_and_volatility_scaling**  
   Use predictive relationships in return signals.

2. **03_07_cross_sectional_alpha_features**  
   Convert lead-lag diagnostics into features.

3. **03_08_tree_models_for_return_prediction**  
   Nonlinear supervised learning for returns.

4. **03_09_information_coefficient_analysis**  
   Evaluate predictive signals robustly.

5. **04_04_covariance_forecasting_dynamic_correlation**  
   Move from mean dynamics to covariance dynamics.

6. **05_01_vectorized_backtest_engine**  
   Test lead-lag signals with realistic costs.

7. **07_01_multi_asset_cta_research_pipeline**  
   Use cross-asset lead-lag effects in a futures strategy.

## 26. Summary

This notebook implemented VAR and Granger causality analysis.

It showed how to:

1. simulate stationary cross-asset VAR data;
2. build lagged VAR design matrices;
3. estimate VAR coefficients by OLS;
4. select lag order with AIC and BIC;
5. forecast out of sample;
6. compare VAR with univariate AR baselines;
7. run Granger causality tests;
8. recover planted lead-lag relationships;
9. study rolling Granger stability;
10. compute impulse responses;
11. build a simplified variance decomposition;
12. save outputs and diagnostics.

The key computational takeaway:

> VAR is a transparent linear forecasting system, but lag design, stationarity, and out-of-sample validation determine whether it is useful.

The key financial takeaway:

> Granger causality is best treated as evidence of incremental predictive content, not proof of true economic causation.

## 27. Further reading

- Hamilton, *Time Series Analysis.*
- Lütkepohl, *New Introduction to Multiple Time Series Analysis.*
- Granger, "Investigating Causal Relations by Econometric Models and Cross-spectral Methods."
- Sims, "Macroeconomics and Reality."
- Literature on Bayesian VAR, sparse VAR, structural VAR, nonlinear Granger causality, and transfer entropy.